# Sprint E3 walkthrough: XS-v1, the cross-sectional model and the risk decomposition

Reproduce every number this sprint stores, by hand where it can be done by
hand, and assert each figure against the artifact it came from.

The three research questions of the sprint:

- Academic. How can cross-sectional equity returns be decomposed into
  systematic factor exposures and idiosyncratic returns, and are the factor
  premia (Fama-MacBeth) distinguishable from zero?
- Practitioner. If I hold this portfolio, how much of my risk is actually
  coming from unintended factor exposures, and which position reduces risk
  fastest if trimmed?
- Research. Does the proposed risk model explain realized portfolio returns
  and volatility adequately enough to support portfolio decisions?

Intuition. The time-series model of E2 asks how a stock moves with the market.
This model asks a different question: on a given day, do small stocks beat
large stocks, and by how much once every other characteristic is held fixed.
The daily regression coefficient is the return of a portfolio with unit
exposure to that characteristic and zero to every other, so factor returns are
tradeable objects. Everything downstream, the covariance, the attribution and
the bias test, is built on that one regression.

Rule for this notebook: no stored number is typed by hand. Every figure is
read from a parquet artifact or from sprints/E3/RESULTS.json, printed, and
asserted. The closing cell collects every stored value and asserts that none
of them appears as a literal in any code cell.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from efb import build, risk
from efb.models import fundamental as fx

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
DATA = ROOT / "data"

# the repository root is importable so the dashboard module, which
# lives outside the package, can be imported by the last section
sys.path.insert(0, str(ROOT))


class _Approx:
    """A stand-in for pytest.approx so the notebook needs no test import."""

    def __init__(self, expected, rel):
        self.expected = float(expected)
        self.rel = rel

    def __eq__(self, other):
        other = float(other)
        scale = max(abs(self.expected), 1e-12)
        if abs(other - self.expected) > self.rel * scale:
            raise AssertionError(f"{other!r} is not within {self.rel} of {self.expected!r}")
        return True

    def __repr__(self):
        return f"approx({self.expected!r}, rel={self.rel})"


def approx(expected, rel=1e-9):
    """Compared against with ==, the way pytest.approx is used."""
    return _Approx(expected, rel)


def read(rel):
    frame = pd.read_parquet(DATA / rel)
    print(f"{rel}: {frame.shape[0]} rows, {len(frame.columns)} columns")
    return frame


version = json.loads((DATA / "VERSION.json").read_text())
registry = json.loads((DATA / "models" / "registry.json").read_text())
results = json.loads((ROOT / "sprints" / "E3" / "RESULTS.json").read_text())
entry = registry["models"]["XS-v1"]

print("data_hash          ", version["data_hash"])
print("XS-v1 artifacts_hash", entry["parameters"]["artifacts_hash"])
print("E3 results hash     ", results["data_hash"])
assert version["data_hash"] == results["data_hash"], "results and manifest differ"
assert entry["parameters"]["artifacts_hash"] != version["data_hash"]  # a subset
print("criteria stored:", sorted(results["criteria"]))
assert set(results["criteria"]) == {f"F3.{i}" for i in range(1, 10)}

descriptors = read("models/XS-v1/descriptors.parquet")
factor_returns = read("models/XS-v1/factor_returns.parquet")
specific = read("models/XS-v1/specific_returns.parquet")
xs_r2 = read("models/XS-v1/xs_r2.parquet")
fmp = read("models/XS-v1/fmp_weights.parquet")
factor_cov = read("models/XS-v1/factor_cov.parquet")
specific_var = read("models/XS-v1/specific_var.parquet")
premia = read("eval/xs_fm_premia.parquet")
decomposition = read("eval/xs_risk_decomposition.parquet")
bias = read("eval/xs_bias.parquet")
exposure = read("eval/xs_exposure_timeseries.parquet")
residual = read("eval/xs_residual_covariance.parquet")
market_cap = read("processed/market_cap.parquet")
bias_by_exposure_table = read("eval/xs_bias_by_exposure.parquet")
coverage_table = read("eval/xs_coverage_by_year.parquet")
survivor_table = read("eval/xs_survivor_restriction.parquet")

data_hash           237c1183b898a054118c1dd0e7fbfa0556b9162841d9eb7fe242da7fa14d2b36
XS-v1 artifacts_hash 87d4d6af68bc4ce60c791405a954b876b66c41ee3e34349d619a93329ec58e83
E3 results hash      237c1183b898a054118c1dd0e7fbfa0556b9162841d9eb7fe242da7fa14d2b36
criteria stored: ['F3.1', 'F3.2', 'F3.3', 'F3.4', 'F3.5', 'F3.6', 'F3.7', 'F3.8', 'F3.9']
models/XS-v1/descriptors.parquet: 664146 rows, 9 columns
models/XS-v1/factor_returns.parquet: 70938 rows, 8 columns
models/XS-v1/specific_returns.parquet: 1842865 rows, 3 columns
models/XS-v1/xs_r2.parquet: 3941 rows, 7 columns


models/XS-v1/fmp_weights.parquet: 1032745 rows, 5 columns
models/XS-v1/factor_cov.parquet: 18 rows, 18 columns
models/XS-v1/specific_var.parquet: 88455 rows, 7 columns
eval/xs_fm_premia.parquet: 72 rows, 8 columns
eval/xs_risk_decomposition.parquet: 180182 rows, 16 columns
eval/xs_bias.parquet: 282 rows, 12 columns
eval/xs_exposure_timeseries.parquet: 6804 rows, 6 columns
eval/xs_residual_covariance.parquet: 282 rows, 9 columns
processed/market_cap.parquet: 2104886 rows, 7 columns
eval/xs_bias_by_exposure.parquet: 6 rows, 12 columns
eval/xs_coverage_by_year.parquet: 24 rows, 11 columns
eval/xs_survivor_restriction.parquet: 17 rows, 9 columns


## 1. The universe and the descriptors

The model runs from the first session on which at least 300 names carry every
descriptor. Momentum is the binding warm-up: the 12-1 window needs 231
observations ending at t-21, so the first complete cross-section is a year
after the panel starts.

In [2]:
print("first session in the panel  ", xs_r2['date'].min().date())
print("first complete cross-section", registry['models']['XS-v1']['parameters']['model_start'])
print("last session                ", xs_r2['date'].max().date())
print("sessions                    ", len(xs_r2))
print("names per session, mean     ", round(float(xs_r2['n_names'].mean()), 2))
print("names per session, minimum  ", int(xs_r2['n_names'].min()))
print("descriptors                 ", registry['models']['XS-v1']['parameters']['descriptors'])
print("factors                     ", registry['models']['XS-v1']['parameters']['n_factors'])
assert registry['models']['XS-v1']['parameters']['n_factors'] == 18
assert int(xs_r2['n_names'].min()) > 300
assert xs_r2['date'].min() >= pd.Timestamp('2011-01-03')

first session in the panel   2011-01-03
first complete cross-section 2011-01-03
last session                 2026-09-03
sessions                     3941
names per session, mean      467.61
names per session, minimum   417
descriptors                  ['market', 'size', 'beta', 'momentum', 'reversal', 'resid_vol', 'liquidity']
factors                      18


In [3]:
date = descriptors['date'].max()
names = ['AAPL', 'XOM', 'JPM']
rows = descriptors.loc[(descriptors['date'] == date) & descriptors['ticker'].isin(names)]
pivot = rows.pivot_table(index='descriptor', columns='ticker',
                         values=['value_raw', 'value_winsor', 'value_z', 'value_z_orth'])
print(f"descriptor values on {date.date()}")
print(pivot.round(6).to_string())
# the standardized value is bounded by the winsorization, the raw one is not
for ticker in names:
    raw = rows.loc[rows['ticker'] == ticker, 'value_raw'].abs().max()
    z = rows.loc[rows['ticker'] == ticker, 'value_z'].abs().max()
    print(f"{ticker}: max |raw| {raw:.4f}, max |z| {z:.4f}")
assert float(rows['value_z'].abs().max()) < 6.0

descriptor values on 2026-09-03
            value_raw                       value_winsor                         value_z                     value_z_orth                    
ticker           AAPL        JPM        XOM         AAPL        JPM        XOM      AAPL       JPM       XOM         AAPL       JPM       XOM
descriptor                                                                                                                                   
beta         0.660575   0.729060  -0.535012     0.660575   0.729060  -0.535012 -0.446367 -0.340110 -2.301375    -0.446367 -0.340110 -2.301375
liquidity   23.546495  21.759788  21.567709    21.540683  21.540683  21.540683  0.572935  0.572935  0.572935     0.572935  0.572935  0.572935
market       1.000000   1.000000   1.000000     1.000000   1.000000   1.000000  1.000000  1.000000  1.000000     1.000000  1.000000  1.000000
momentum     0.307837   0.222372   0.385228     0.307837   0.222372   0.385228  0.155874 -0.133140  0.417585     0.0

In [4]:
# The standardized market factor is the constant column, and the cap-weighted
# mean style exposure is zero by construction. Both are checked here from the
# artifact, not from the code that produced it. The weights are the market caps
# dated t-1, which is the vintage the standardizer used.
size = descriptors.loc[
    (descriptors['descriptor'] == 'size') & descriptors['value_z'].notna()
]
lagged_cap = market_cap.pivot(
    index='date', columns='ticker', values='market_cap'
).shift(1)
weights = lagged_cap.stack().rename('cap_lag').reset_index()
weights.columns = ['date', 'ticker', 'cap_lag']
joined = size.merge(weights, on=['date', 'ticker'], how='left')
joined = joined.dropna(subset=['cap_lag'])
weighted = joined.groupby('date').apply(
    lambda block: np.average(block['value_z'], weights=block['cap_lag']),
    include_groups=False,
)
print("cap-weighted mean of the size z score, over dates")
print(f"  mean {weighted.mean():.3e}, max abs {weighted.abs().max():.3e}, "
      f"dates {len(weighted)}")
assert float(weighted.abs().max()) < 1e-9


cap-weighted mean of the size z score, over dates
  mean -1.063e-16, max abs 8.158e-15, dates 189


## 2. The regime, by hand

One cross-section, built from the artifact and the panel, fitted by hand. The
design has seven style columns and ten sector dummies: the reference sector has
none, because a market column plus eleven dummies is collinear. Its factor
return is derived from the constraint instead.

In [5]:
inputs = __import__('efb.probes', fromlist=['load_panel']).load_panel(DATA)
returns, close, volume = inputs['returns'], inputs['close'], inputs['volume']
sectors, mapped, shares = inputs['sectors'], inputs['mapped'], inputs['shares_mapped']
cap = fx.market_cap(close[mapped], shares)
proxy = fx.market_proxy(returns[mapped], cap)
design = fx.build_design(returns=returns[mapped], close=close[mapped], volume=volume[mapped],
                         market_cap=cap, sectors=sectors, proxy=proxy)
print("design days", len(design.days), "reported factors", len(design.factor_names),
      "estimated columns", fx.N_ESTIMATED)
print("reference sector", fx.REFERENCE_SECTOR)
last = design.days[-1]
print("last cross-section", last.date.date(), last.tickers.size, "names")
print("estimated design", last.design.shape, "reported design", last.risk_design.shape)
assert last.design.shape[1] == fx.N_ESTIMATED
assert last.risk_design.shape[1] == len(fx.FACTOR_NAMES)

design days 3941 reported factors 18 estimated columns 17
reference sector Real Estate
last cross-section 2026-09-03 497 names
estimated design (497, 17) reported design (497, 18)


In [6]:
fit = fx.wls_fit(last.design, last.returns, last.weights)
weights = fx.sector_cap_weights(last)
identified = fx.identify(fit.factor_returns, weights, n_styles=len(fx.STYLE_NAMES))

stored = factor_returns.loc[factor_returns['date'] == last.date].set_index('factor')
print(f"factor returns on {last.date.date()}")
print(pd.DataFrame({
    'by hand (identified)': pd.Series(identified.as_vector(), index=fx.FACTOR_NAMES),
    'artifact': stored['f'],
    'artifact pre-identification': stored['f_pre_identification'],
}).round(8).to_string())
assert np.allclose(identified.as_vector(), stored['f'].to_numpy(), atol=1e-12)
print("hand-computed factor returns match the artifact to 1e-12")


factor returns on 2026-09-03
           by hand (identified)  artifact  artifact pre-identification
market                 0.011677  0.011677                     0.025666
size                   0.001238  0.001238                     0.001238
beta                   0.005774  0.005774                     0.005774
momentum              -0.002689 -0.002689                    -0.002689
reversal               0.006897  0.006897                     0.006897
resid_vol              0.001953  0.001953                     0.001953
liquidity              0.001058  0.001058                     0.001058
sector_10             -0.014815 -0.014815                    -0.028803
sector_15             -0.011800 -0.011800                    -0.025789
sector_20              0.007682  0.007682                    -0.006307
sector_25              0.003279  0.003279                    -0.010710
sector_30             -0.000343 -0.000343                    -0.014332
sector_35             -0.007347 -0.007347       

In [7]:
# Which sector is the reference one, and why the market factor is the
# cap-weighted market return
print("sector cap weights on the day")
print(pd.Series(weights, index=fx.SECTOR_NAMES).round(6).to_string())
print(f"cap-weighted sector sum of the reported returns: "
      f"{float(identified.sector @ weights):+.3e}")
print(f"market factor (reported)  {identified.market:+.8f}")
print(f"cap-weighted universe return, from returns and t-1 market caps:")
day_return = returns.loc[last.date, last.tickers]
day_weights = cap.shift(1).loc[last.date, last.tickers]
print(f"  {float(np.average(day_return, weights=day_weights)):+.8f}")
assert abs(float(identified.sector @ weights)) < 1e-15

sector cap weights on the day
Energy                    0.032568
Materials                 0.016760
Industrials               0.074444
Consumer Discretionary    0.091397
Consumer Staples          0.047575
Health Care               0.087290
Financials                0.117809
Information Technology    0.342010
Communication Services    0.155360
Utilities                 0.018326
Real Estate               0.016462
cap-weighted sector sum of the reported returns: +1.518e-18
market factor (reported)  +0.01167668
cap-weighted universe return, from returns and t-1 market caps:
  +0.01145754


## 3. The factor-mimicking portfolios

The rows of (X'WX)^-1 X'W are portfolios with unit exposure to their own factor
and zero to every other. This is F3.2, checked on the stored weights as well as
by hand.

In [8]:
estimated_weights = fmp.loc[(fmp['date'] == last.date) & (fmp['kind'] == 'estimated')]
matrix = (estimated_weights.pivot_table(index='factor', columns='ticker', values='weight')
          .reindex(index=list(fx.ESTIMATED_NAMES), columns=last.tickers))
# named fmp_exposure rather than exposure, because `exposure` is the stored
# exposure time series the later sections read
fmp_exposure = last.design.T @ matrix.to_numpy().T
deviation = np.abs(fmp_exposure - np.eye(fx.N_ESTIMATED))
print(f"max abs deviation from the unit matrix: {deviation.max():.3e}")
print("max deviation per factor")
print(pd.Series(deviation.max(axis=1), index=fx.ESTIMATED_NAMES).to_string())
assert deviation.max() < 1e-8
print("F3.2 holds on the stored weights")

fmp_return = matrix.to_numpy() @ last.returns
print(f"FMP row times returns vs the estimated factor return: "
      f"{float(np.max(np.abs(fmp_return - fit.factor_returns))):.3e}")
assert np.allclose(fmp_return, fit.factor_returns, atol=1e-12)


max abs deviation from the unit matrix: 3.153e-14
max deviation per factor
market       2.686740e-14
size         3.971209e-15
beta         5.603974e-15
momentum     4.540838e-15
reversal     2.659895e-14
resid_vol    3.749879e-15
liquidity    3.153033e-14
sector_10    1.554312e-15
sector_15    1.385177e-15
sector_20    1.805847e-15
sector_25    4.543458e-15
sector_30    2.220446e-15
sector_35    1.733639e-15
sector_40    2.866197e-15
sector_45    1.098514e-14
sector_50    3.026225e-15
sector_55    2.775558e-15
F3.2 holds on the stored weights
FMP row times returns vs the estimated factor return: 3.469e-16


In [9]:
# the identity is checked every day by the build, and the daily number is stored
worst = xs_r2.loc[xs_r2['fmp_identity_max_abs_error'].idxmax()]
print(f"worst identity error over {len(xs_r2)} sessions: "
      f"{float(worst['fmp_identity_max_abs_error']):.3e} on {worst['date'].date()}")
print(f"worst cap-weighted sector sum: {float(xs_r2['sector_cap_weighted_sum'].abs().max()):.3e}")
print("F3.2 stored number:", results['criteria']['F3.2']['stored_numbers']['max_abs_identity_error_estimated'])
assert float(worst['fmp_identity_max_abs_error']) < 1e-8
assert float(xs_r2['sector_cap_weighted_sum'].abs().max()) < 1e-10

worst identity error over 3941 sessions: 1.243e-13 on 2026-06-11
worst cap-weighted sector sum: 7.047e-18
F3.2 stored number: 1.2426149519073615e-13


## 4. The cross-sectional R squared

F3.1 asks for an average above 20 percent, with a fail band below 15. The three
diagnostics standing instruction B asks for are stored whether or not the
average clears the bar.

In [10]:
parameters = entry['parameters']
print(f"average R squared   {float(xs_r2['r_squared'].mean()):.6f}")
print(f"by year             {parameters['r_squared_by_year']}")
print(f"market only         {parameters['r_squared_market_only_mean']} "
      "(zero by construction: the market factor is the intercept)")
print(f"sectors only        {parameters['r_squared_sectors_only_mean']}")
print(f"styles only         {parameters['r_squared_styles_only_mean']}")
print(f"by sector           {parameters['r_squared_by_sector']}")
print("F3.1 stored:", results['criteria']['F3.1']['stored_numbers']['mean_cross_sectional_r_squared'])
assert abs(float(xs_r2['r_squared'].mean()) - results['criteria']['F3.1']['stored_numbers']['mean_cross_sectional_r_squared']) < 1e-12

average R squared   0.329450
by year             {'2011': 0.35381, '2012': 0.289056, '2013': 0.247919, '2014': 0.28293, '2015': 0.293861, '2016': 0.31435, '2017': 0.277233, '2018': 0.312246, '2019': 0.28363, '2020': 0.37178, '2021': 0.340566, '2022': 0.414612, '2023': 0.367565, '2024': 0.3654, '2025': 0.380905, '2026': 0.398301}
market only         -0.0 (zero by construction: the market factor is the intercept)
sectors only        0.192787
styles only         0.200683
by sector           {'Communication Services': 0.312029, 'Consumer Discretionary': 0.227468, 'Consumer Staples': 0.27365, 'Energy': 0.389566, 'Financials': 0.271331, 'Health Care': 0.217544, 'Industrials': 0.244622, 'Information Technology': 0.258247, 'Materials': 0.19037, 'Real Estate': 0.316339, 'Utilities': 0.435051}
F3.1 stored: 0.3294501878861149


In [11]:
# The shift test, printed both ways. X_(t-1) is the design the model uses:
# every descriptor computed from data through t-1. X_t is the design one day
# later, dated t, so it contains same-day information. Both are asked to
# explain the same returns r_t on the same names, which is the only way the
# two numbers are comparable.
by_date = {day.date: day for day in design.days}
dates = sorted(by_date)
lagged, dated_t = [], []
for position, date in enumerate(dates[:-1]):
    day, later = by_date[date], by_date[dates[position + 1]]
    common = day.tickers.intersection(later.tickers)
    if len(common) < fx.MIN_NAMES:
        continue
    here = day.tickers.get_indexer(common)
    there = later.tickers.get_indexer(common)
    target = day.returns[here]
    ok = np.isfinite(target)
    lagged.append(
        fx.wls_fit(day.design[here][ok], target[ok], day.weights[here][ok]).r_squared
    )
    dated_t.append(
        fx.wls_fit(later.design[there][ok], target[ok], later.weights[there][ok]).r_squared
    )

mean_lagged = float(np.mean(lagged))
mean_dated_t = float(np.mean(dated_t))
print(f"X_(t-1) on r_t, {len(lagged)} days: mean R squared {mean_lagged:.6f}")
print(f"X_t     on r_t, {len(dated_t)} days: mean R squared {mean_dated_t:.6f}")
print(f"difference, the same-day information the lag removes: "
      f"{mean_dated_t - mean_lagged:+.6f}")
assert np.isfinite(lagged).all() and np.isfinite(dated_t).all()
assert mean_dated_t > mean_lagged, "the dated-t design must carry same-day information"
assert abs(mean_lagged - float(xs_r2['r_squared'].mean())) < 5e-4
print("the dated-t design explains more, which is what the t-1 rule removes, and")
print("the lagged number reproduces the stored R squared: the rule holds in the build")

X_(t-1) on r_t, 3940 days: mean R squared 0.329462
X_t     on r_t, 3940 days: mean R squared 0.364005
difference, the same-day information the lag removes: +0.034543
the dated-t design explains more, which is what the t-1 rule removes, and
the lagged number reproduces the stored R squared: the rule holds in the build


## 4a. Why the shift test came out the other way

INPUT: `models/XS-v1/descriptors.parquet` (date, ticker, descriptor, value_raw),
`processed/returns.parquet` (date, ticker, r) and `processed/market_cap.parquet`
(date, ticker, close, shares, market_cap), plus the stored shift-test pair in
`sprints/E3/RESULTS.json` (criteria.F3.1.stored_numbers).

The PRD's design rule says a shift test proves that replacing X_(t-1) with X_t
lowers the cross-sectional R squared. The measurement says the opposite, and the
reason is mechanical rather than statistical. For a date t, look at which input
window each descriptor uses when it is built at t-1 and when it is built at t:

| Descriptor | Window in X_(t-1), the design at t | Window in X_t, the design at t+1 | Contains r_t? |
| --- | --- | --- | --- |
| size | log market cap at t-1 | log market cap at t | yes, cap at t is cap at t-1 times one plus r_t |
| beta | 252 sessions ending t-1 | 252 sessions ending t | yes |
| momentum | 231 sessions ending 21 sessions before t | 231 sessions ending 20 sessions before t | no |
| reversal | 20 sessions ending t-1 | 20 sessions ending t | yes |
| resid_vol | 63 sessions ending t-1 | 63 sessions ending t | yes |
| liquidity | 63 sessions ending t-1 | 63 sessions ending t | yes |

Five of the seven style columns end their window on the date being explained,
and the size column contains the return directly: market capitalisation at t-1
is market capitalisation at t-2 times one plus r_t, so the later design has the
regressand inside its own regressors. That is the definition of the leakage the
t-1 rule exists to prevent, so a higher R squared for X_t is what a correct
implementation must show rather than a bug. The PRD's prediction was wrong about
the sign and the model is right; the lag is load bearing precisely because it
costs 3.45 points of fit, and the two cells below prove the mechanism name by
name and then price it in R squared.


In [12]:
# INPUT: the design built above, the panel returns from data/processed/returns.parquet,
#        data/models/XS-v1/descriptors.parquet (date, ticker, descriptor, value_raw),
#        data/processed/market_cap.parquet (date, ticker, close, shares, market_cap)
# OUTPUT: none, printed and asserted only
ticker = 'AAPL'
# the last two cross-sections, so every window below is fully populated
earlier, later = design.days[-2], design.days[-1]
assert ticker in earlier.tickers and ticker in later.tickers
raw = design.raw
log_returns = np.log1p(returns)

# The artifact's raw reversal and momentum, recomputed by hand from the panel, so
# the window ends are established rather than asserted in prose. Both windows are
# anchored on the X_t row, which is the design the dated-t fit uses.
window = log_returns.loc[:later.date, ticker]
# reversal at X_t: 20 sessions ending on the previous session, which is the date
# being explained, so this window contains r_t
reversal_block = window.iloc[-21:-1]
reversal_hand = np.expm1(reversal_block.sum())
# momentum at X_t: 231 sessions ending 21 sessions before X_t, which is 20
# sessions before the date being explained, so this window does not contain r_t
momentum_block = window.iloc[-252:-21]
momentum_hand = np.expm1(momentum_block.sum())
print(f"{ticker} at {earlier.date.date()} and the next cross-section {later.date.date()}")
print(f"  reversal, artifact at X_(t-1) {float(raw['reversal'].loc[earlier.date, ticker]):+.6f}")
print(f"  reversal, artifact at X_t     {float(raw['reversal'].loc[later.date, ticker]):+.6f}")
print(f"  reversal, hand sum of 20 sessions ending {earlier.date.date()}: {float(reversal_hand):+.6f}")
print(f"  momentum, artifact at X_(t-1) {float(raw['momentum'].loc[earlier.date, ticker]):+.6f}")
print(f"  momentum, artifact at X_t     {float(raw['momentum'].loc[later.date, ticker]):+.6f}")
print(f"  momentum, hand sum of {len(momentum_block)} sessions from "
      f"{momentum_block.index[0].date()} to {momentum_block.index[-1].date()}: "
      f"{float(momentum_hand):+.6f}")
assert float(raw['reversal'].loc[later.date, ticker]) == approx(
    float(reversal_hand), rel=1e-9
), "the later reversal window ends on the date being explained"
assert float(raw['momentum'].loc[later.date, ticker]) == approx(
    float(momentum_hand), rel=1e-9
), "the later momentum window ends 21 sessions earlier"

# the size column is the leak in its purest form: its one day change is the
# log of the price ratio, which is the return being explained
cap_close = market_cap.set_index(['date', 'ticker'])['close']
shares_now = market_cap.set_index(['date', 'ticker'])['shares']
size_change = float(raw['size'].loc[later.date, ticker] - raw['size'].loc[earlier.date, ticker])
explained = float(np.log1p(returns.loc[earlier.date, ticker]))
print()
print(f"  size, artifact at X_(t-1)     {float(raw['size'].loc[earlier.date, ticker]):+.6f}")
print(f"  size, artifact at X_t         {float(raw['size'].loc[later.date, ticker]):+.6f}")
print(f"  size change X_t minus X_(t-1) {size_change:+.6f}")
print(f"  log of one plus the return    {explained:+.6f}")
print(f"  close {float(cap_close.loc[(earlier.date, ticker)]):.4f} to "
      f"{float(cap_close.loc[(later.date, ticker)]):.4f}, shares "
      f"{float(shares_now.loc[(earlier.date, ticker)]):.0f} to "
      f"{float(shares_now.loc[(later.date, ticker)]):.0f}")
assert size_change == approx(explained, rel=1e-9), (
    "the later size descriptor contains the return being explained"
)
print()
print("So X_t fits better because five of its seven style columns end their window")
print("on the date being explained, and the size column is that date's return in")
print("the regressors. This is the leakage the t-1 rule removes, not a defect.")


AAPL at 2026-09-02 and the next cross-section 2026-09-03
  reversal, artifact at X_(t-1) +0.051815
  reversal, artifact at X_t     +0.045789
  reversal, hand sum of 20 sessions ending 2026-09-02: +0.045789
  momentum, artifact at X_(t-1) +0.350580
  momentum, artifact at X_t     +0.307837
  momentum, hand sum of 231 sessions from 2025-09-04 to 2026-08-05: +0.307837



  size, artifact at X_(t-1)     +29.188114
  size, artifact at X_t         +29.187591
  size change X_t minus X_(t-1) -0.000523
  log of one plus the return    -0.000523
  close 324.9600 to 328.2100, shares 14594180000 to 14594180000

So X_t fits better because five of its seven style columns end their window
on the date being explained, and the size column is that date's return in
the regressors. This is the leakage the t-1 rule removes, not a defect.


In [13]:
# INPUT: the design built above and the stored shift-test pair in
#        sprints/E3/RESULTS.json (criteria.F3.1.stored_numbers)
# OUTPUT: none, printed and asserted only
styles = {name: index for index, name in enumerate(fx.STYLE_NAMES)}
leaking = [styles['size'], styles['beta'], styles['reversal'],
           styles['resid_vol'], styles['liquidity']]
keep_clean = [index for index in range(last.design.shape[1]) if index not in leaking]

r2_lagged, r2_dated_t, r2_no_size, r2_clean = [], [], [], []
for position, date in enumerate(dates[:-1]):
    day, later = by_date[date], by_date[dates[position + 1]]
    common = day.tickers.intersection(later.tickers)
    if len(common) < fx.MIN_NAMES:
        continue
    here = day.tickers.get_indexer(common)
    there = later.tickers.get_indexer(common)
    target = day.returns[here]
    ok = np.isfinite(target)
    if int(ok.sum()) < fx.MIN_NAMES:
        continue
    r2_lagged.append(fx.wls_fit(day.design[here][ok], target[ok], day.weights[here][ok]).r_squared)
    r2_dated_t.append(fx.wls_fit(later.design[there][ok], target[ok], later.weights[there][ok]).r_squared)
    r2_no_size.append(
        fx.wls_fit(
            np.delete(later.design[there][ok], styles['size'], axis=1),
            target[ok],
            later.weights[there][ok],
        ).r_squared
    )
    r2_clean.append(
        fx.wls_fit(
            later.design[there][ok][:, keep_clean], target[ok], later.weights[there][ok]
        ).r_squared
    )

means = {
    'X_(t-1) on r_t, the model': float(np.mean(r2_lagged)),
    'X_t on r_t, everything': float(np.mean(r2_dated_t)),
    'X_t on r_t, size dropped': float(np.mean(r2_no_size)),
    'X_t on r_t, only the clean columns': float(np.mean(r2_clean)),
}
for label, value in means.items():
    print(f"  {label:36s} {value:.6f}")
print(f"  {'gain from using X_t':36s} {means['X_t on r_t, everything'] - means['X_(t-1) on r_t, the model']:+.6f}")
print(f"  {'of which the size column alone':36s} "
      f"{means['X_t on r_t, everything'] - means['X_t on r_t, size dropped']:+.6f}")

stored = results['criteria']['F3.1']['stored_numbers']
print()
print(f"stored lagged   {stored['shift_test_lagged_mean_r_squared']:.6f}")
print(f"stored dated t  {stored['shift_test_dated_t_mean_r_squared']:.6f}")
assert means['X_(t-1) on r_t, the model'] == approx(
    stored['shift_test_lagged_mean_r_squared'], rel=1e-9
)
assert means['X_t on r_t, everything'] == approx(
    stored['shift_test_dated_t_mean_r_squared'], rel=1e-9
)
assert means['X_t on r_t, everything'] > means['X_(t-1) on r_t, the model']
assert means['X_t on r_t, size dropped'] < means['X_t on r_t, everything']
assert means['X_t on r_t, only the clean columns'] < means['X_t on r_t, everything']
print()
print("the notebook reproduces both stored numbers, and most of the gain comes from")
print("the columns whose window ends on the date being explained")


  X_(t-1) on r_t, the model            0.329462
  X_t on r_t, everything               0.364005
  X_t on r_t, size dropped             0.359079
  X_t on r_t, only the clean columns   0.239774
  gain from using X_t                  +0.034543
  of which the size column alone       +0.004926

stored lagged   0.329462
stored dated t  0.364005

the notebook reproduces both stored numbers, and most of the gain comes from
the columns whose window ends on the date being explained


## 4b. The Market factor, estimated both ways

Standing instruction C: the Market factor estimated on total returns and on
excess returns, side by side, with their correlation on the overlap through
2026-07-31. XS-v1 uses the total return because the panel's excess column is
all NaN after that date. The design is fitted twice on the same days, once on
r and once on r - rf, and the difference is measured rather than assumed.


In [14]:
ff = pd.read_parquet(DATA / 'raw' / 'factors_ff.parquet')
rf = ff['rf'].astype(float)
print(f"risk-free series: {len(rf)} rows, {rf.index.min().date()} to {rf.index.max().date()}")

sample = design.days[-250:]
days = [day.date for day in sample]
total, excess, gaps, style_gaps = [], [], [], []
for day in sample:
    rate = rf.get(day.date, np.nan)
    if not np.isfinite(rate):
        continue
    base = fx.wls_fit(day.design, day.returns, day.weights)
    shifted = fx.wls_fit(day.design, day.returns - rate, day.weights)
    cap_weights = fx.sector_cap_weights(day)
    a = fx.identify(base.factor_returns, cap_weights, n_styles=len(fx.STYLE_NAMES))
    b = fx.identify(shifted.factor_returns, cap_weights, n_styles=len(fx.STYLE_NAMES))
    total.append(a.market)
    excess.append(b.market)
    gaps.append(a.market - b.market - rate)
    style_gaps.append(np.max(np.abs(a.as_vector()[1:] - b.as_vector()[1:])))

total = np.array(total)
excess = np.array(excess)
print(f"days fitted both ways            {len(total)}")
print(f"market, total returns, mean      {total.mean():.8f}")
print(f"market, excess returns, mean     {excess.mean():.8f}")
print(f"risk-free mean on the same days  {rf.reindex(days).mean():.8f}")
print(f"correlation of the two series    {np.corrcoef(total, excess)[0, 1]:.10f}")
print(f"max abs (gap - rf)               {np.max(np.abs(gaps)):.3e}")
print(f"max abs style and sector gap     {np.max(np.abs(style_gaps)):.3e}")
assert np.max(np.abs(gaps)) < 1e-12, "a common shift must move only the intercept"
assert np.max(np.abs(style_gaps)) < 1e-12, "the other factors must not move at all"
assert np.corrcoef(total, excess)[0, 1] > 0.999
print("the two estimators differ by exactly the risk-free rate, so the regressand")
print("choice is a mean shift and the two series are correlation 1 for a PM")

# The same comparison against the FF total return, on the full overlap.
market_total = (factor_returns.loc[factor_returns['factor'] == 'market']
                .set_index('date')['f'].sort_index())
ff_total = (ff['mkt_rf'] + ff['rf']).reindex(market_total.index).dropna()
joined = pd.concat([market_total, ff_total], axis=1, join='inner').dropna()
correlation = float(joined.iloc[:, 0].corr(joined.iloc[:, 1]))
print(f"\nmodel market factor vs FF total return, {len(joined)} days: {correlation:.10f}")
print("stored F3.4 (market):",
      results['criteria']['F3.4']['stored_numbers']['market_vs_ff_market_correlation'])
assert abs(correlation - results['criteria']['F3.4']['stored_numbers']['market_vs_ff_market_correlation']) < 1e-10


risk-free series: 4169 rows, 2010-01-04 to 2026-07-31
days fitted both ways            226
market, total returns, mean      0.00094302
market, excess returns, mean     0.00077975
risk-free mean on the same days  0.00016327
correlation of the two series    0.9999864065
max abs (gap - rf)               4.990e-16
max abs style and sector gap     8.969e-16
the two estimators differ by exactly the risk-free rate, so the regressand
choice is a mean shift and the two series are correlation 1 for a PM

model market factor vs FF total return, 3917 days: 0.9889255745
stored F3.4 (market): 0.9889255744977894


## 5. The factor covariance and the risk decomposition

Sigma = X F X' + D with F the exponentially weighted factor covariance
(half-life 90, Newey-West lag 2) estimated from factor returns up to the date,
and D the exponentially weighted specific variance (half-life 42) shrunk toward
the (sector, size tercile) bucket mean. The Euler decomposition then gives
sigma_p, the marginal contributions and the factor contributions.

In [15]:
factor_wide = (factor_returns.pivot(index='date', columns='factor', values='f')
               .reindex(columns=list(fx.FACTOR_NAMES)).dropna())
specific_wide = specific.pivot(index='date', columns='ticker', values='specific_return')
as_of = pd.Timestamp('2026-08-31') if pd.Timestamp('2026-08-31') in factor_wide.index else factor_wide.index[-1]
covariance = fx.ewma_factor_cov(factor_wide.loc[:as_of], half_life=fx.F_HALF_LIFE)
print(f"factor covariance as of {as_of.date()}, {covariance.shape}")
print(covariance.round(8).to_string())
eigenvalues = np.linalg.eigvalsh(covariance.to_numpy())
print(f"eigenvalues: min {eigenvalues.min():.3e}, max {eigenvalues.max():.3e}")
assert eigenvalues.min() > -1e-14

factor covariance as of 2026-08-31, (18, 18)
factor           market          size      beta  momentum  reversal     resid_vol  liquidity     sector_10  sector_15  sector_20  sector_25  sector_30     sector_35     sector_40  sector_45  sector_50  sector_55  sector_60
factor                                                                                                                                                                                                                        
market     8.926000e-05  9.730000e-06  0.000056  0.000004 -0.000003  1.888000e-05  -0.000001 -4.004000e-05  -0.000002  -0.000004   0.000009  -0.000009 -2.422000e-05 -8.000000e-08   0.000004   0.000010   0.000005   0.000007
size       9.730000e-06  3.615000e-05  0.000022  0.000009 -0.000001  1.572000e-05  -0.000029  5.490000e-06   0.000007   0.000005  -0.000001   0.000008 -7.700000e-07  8.030000e-06   0.000004  -0.000025   0.000034   0.000016
beta       5.600000e-05  2.182000e-05  0.000069  0.000008  0.00

In [16]:
days = {day.date: day for day in design.days}
day = days[as_of]
variance = specific_var.loc[specific_var['date'] == as_of].set_index('ticker')['specific_var']
book = risk.load_book(DATA / 'portfolios' / 'seed_mom_ls.parquet', as_of)
result = risk.decompose('seed_mom_ls', as_of, book, day, covariance, variance)

# the same decomposition against an explicitly formed Sigma
design_matrix = day.risk_design
d = variance.reindex(day.tickers).fillna(float(variance.median())).to_numpy()
sigma = design_matrix @ covariance.to_numpy() @ design_matrix.T + np.diag(d)
w = book.reindex(day.tickers).fillna(0.0).to_numpy(dtype=float)
explicit_total = float(w @ sigma @ w)
explicit_mcr = sigma @ w / np.sqrt(explicit_total)
print(f"sigma_p            {result.sigma_p:.6f}   explicit {np.sqrt(explicit_total):.6f}")
print(f"factor variance    {result.factor_variance:.10e}")
print(f"idio variance      {result.idio_variance:.10e}")
print(f"sum                {result.factor_variance + result.idio_variance:.10e}")
print(f"w' Sigma w         {explicit_total:.10e}")
print(f"max |MCR - explicit| {float(np.max(np.abs(result.mcr.to_numpy() - explicit_mcr))):.3e}")
print(f"sum of contributions {float(result.contribution.sum()):.10f} vs sigma_p {result.sigma_p:.10f}")
print(f"residual weight (no descriptor row) {result.residual_weight:.6f}")
assert abs(result.total_variance - explicit_total) < 1e-18
assert abs(float(result.contribution.sum()) - result.sigma_p) < 1e-12
assert float(np.max(np.abs(result.mcr.to_numpy() - explicit_mcr))) < 1e-12

sigma_p            0.004429   explicit 0.004429
factor variance    1.6193874082e-05
idio variance      3.4225745859e-06
sum                1.9616448668e-05
w' Sigma w         1.9616448668e-05
max |MCR - explicit| 5.204e-18
sum of contributions 0.0044290460 vs sigma_p 0.0044290460
residual weight (no descriptor row) -0.006277


In [17]:
print("factor exposures and contributions, momentum book")
factor_rows = decomposition.loc[(decomposition['book'] == 'seed_mom_ls')
                                & (decomposition['date'] == as_of)
                                & (decomposition['level'] == 'factor')]
print(factor_rows[['name', 'exposure']].set_index('name').T.round(4).to_string())
print()
print(f"stored factor variance {float(factor_rows['factor_variance'].iloc[0]):.10e}")
print(f"stored idio variance   {float(factor_rows['idio_variance'].iloc[0]):.10e}")
assert abs(float(factor_rows['factor_variance'].iloc[0]) - result.factor_variance) < 1e-18

factor exposures and contributions, momentum book
name      market    size    beta  momentum  reversal  resid_vol  liquidity  sector_10  sector_15  sector_20  sector_25  sector_30  sector_35  sector_40  sector_45  sector_50  sector_55  sector_60
exposure  0.0063  0.1756 -0.0752     0.635    0.3385    -0.0779    -0.0186        0.0        0.0        0.0        0.0        0.0        0.0      0.003     0.0032        0.0       -0.0        0.0

stored factor variance 1.6193874082e-05
stored idio variance   3.4225745859e-06


In [18]:
print("top 5 marginal contributions to risk, momentum book")
for ticker, value in result.top_mcr:
    print(f"  {ticker}: MCR {value:+.6f}, weight {float(result.weights[ticker]):+.6f}, "
          f"contribution {float(result.contribution[ticker]):+.6f}")
print()
print("top 5 for the equal-weight book")
book_ew = risk.load_book(DATA / 'portfolios' / 'seed_ew.parquet', as_of)
result_ew = risk.decompose('seed_ew', as_of, book_ew, day, covariance, variance)
print(f"  sigma_p {result_ew.sigma_p:.6f}, factor share {result_ew.factor_share:.4f}, "
      f"residual weight {result_ew.residual_weight:.6f}")
for ticker, value in result_ew.top_mcr:
    print(f"  {ticker}: MCR {value:+.6f}, weight {float(result_ew.weights[ticker]):+.6f}")

top 5 marginal contributions to risk, momentum book
  TTD: MCR -0.022407, weight -0.011364, contribution +0.000255
  APP: MCR -0.019127, weight -0.011364, contribution +0.000217
  NKE: MCR -0.014590, weight -0.005051, contribution +0.000074
  NCLH: MCR -0.014564, weight -0.005051, contribution +0.000074
  PODD: MCR -0.014326, weight -0.004132, contribution +0.000059

top 5 for the equal-weight book


  sigma_p 0.006955, factor share 0.9783, residual weight 0.011928
  MRNA: MCR +0.020299, weight +0.001988
  TTD: MCR +0.014982, weight +0.001988
  NCLH: MCR +0.014971, weight +0.001988
  APP: MCR +0.014026, weight +0.001988
  BLDR: MCR +0.013923, weight +0.001988


## 6. The exposure timing problem

This is the E2 open item that this sprint closes. A book that re-sorts on
momentum every month holds a momentum exposure that a full-sample regression
cannot see, because the full-sample name-level betas average out.

In [19]:
mom = exposure.loc[(exposure['book'] == 'seed_mom_ls')
                   & (exposure['factor'] == 'momentum')].set_index('date')['exposure']
print(f"XS-v1 exposure to momentum, {len(mom)} rebalances from {mom.index.min().date()}")
print(f"  mean {mom.mean():+.6f}, min {mom.min():+.6f}, max {mom.max():+.6f}")
print(f"  share of rebalances positive: {float((mom > 0).mean()):.4f}")
stored = results['criteria']['F3.9']['stored_numbers']
print()
print("reconciliation against the E2 measurements")
print(f"  XS-v1 exposure mean (this sprint, from 2015)  {stored['xs_exposure_mean']:+.6f}")
print(f"  E2 rolling-beta aggregate mean                {stored['e2_reference']['rolling_beta_aggregate_mean']:+.6f}")
print(f"  E2 rolling-beta range                         {stored['e2_reference']['rolling_beta_aggregate_min']:+.6f} "
      f"to {stored['e2_reference']['rolling_beta_aggregate_max']:+.6f}")
print(f"  E2 regression loading on MOM                  {stored['e2_reference']['regression_loading']:+.6f} "
      f"(t {stored['e2_reference']['regression_t_stat']:.2f})")
print(f"  E2 static name-level factor share, last month {stored['e2_reference']['static_name_level_factor_share_last_month']:+.6f}")
print(f"  E2 static full-sample aggregate               {stored['e2_reference']['static_full_sample_aggregate']:+.6f}")
assert mom.mean() > 0
assert mom.max() > 0.5

XS-v1 exposure to momentum, 189 rebalances from 2011-01-31
  mean +0.719794, min +0.231959, max +1.091340
  share of rebalances positive: 1.0000

reconciliation against the E2 measurements
  XS-v1 exposure mean (this sprint, from 2015)  +0.721498
  E2 rolling-beta aggregate mean                +0.069780
  E2 rolling-beta range                         -0.332205 to +0.390455
  E2 regression loading on MOM                  +0.288464 (t 29.10)
  E2 static name-level factor share, last month +0.096820
  E2 static full-sample aggregate               -0.016190


In [20]:
# why the two measurements differ: the exposure is a property of the book at
# the moment it is held, and the book's composition changes every month
sizes = exposure.loc[(exposure['book'] == 'seed_mom_ls')
                     & (exposure['factor'] == 'size')].set_index('date')['exposure']
print(f"size exposure, mean {sizes.mean():+.6f}, range {sizes.min():+.6f} to {sizes.max():+.6f}")
print("correlation between the momentum and size exposures of the book:",
      round(float(mom.reindex(sizes.index).corr(sizes)), 4))
print()
print("A book that is long past winners and short past losers carries a")
print("positive momentum exposure whenever it is measured, and the exposure")
print("moves with the dispersion of the momentum signal. A full-sample")
print("regression of the book's returns on MOM averages that exposure toward")
print("zero, which is why the static factor share is 0.0968 while the")
print("measured exposure is positive at most rebalances.")

size exposure, mean +0.088020, range -0.366257 to +0.529736
correlation between the momentum and size exposures of the book: 0.336

A book that is long past winners and short past losers carries a
positive momentum exposure whenever it is measured, and the exposure
moves with the dispersion of the momentum signal. A full-sample
regression of the book's returns on MOM averages that exposure toward
zero, which is why the static factor share is 0.0968 while the
measured exposure is positive at most rebalances.


## 7. Realized residual covariance

F3.8 asks for the factor share of variance both ways: with the diagonal D and
with the realized residual covariance over a trailing window.

In [21]:
print(residual.round(6).to_string(index=False))
summary = residual.groupby('book')[
    ['factor_share_diagonal', 'factor_share_realized']
].mean()
print()
print(summary.round(6).to_string())
print()
higher = residual.assign(
    realized_higher=(
        residual['factor_share_realized'] > residual['factor_share_diagonal']
    )
).groupby('book')['realized_higher'].mean()
print("share of windows where the realized factor share exceeds the diagonal one")
print(higher.round(4).to_string())
print()
print("The direction is measured, not assumed: the diagonal D is an exponentially")
print("weighted specific variance shrunk toward the sector and size bucket mean,")
print("and the realized residual covariance is the sample covariance of the same")
print("specific returns over a trailing year, so the two disagree in either")
print("direction depending on the book.")
print("F3.8 stored numbers:")
print(" ", results['criteria']['F3.8']['stored_numbers'])
assert set(residual['book']) == {'seed_ew', 'seed_mom_ls'}
assert residual[['factor_share_diagonal', 'factor_share_realized']].notna().all().all()
assert float(higher.loc['seed_mom_ls']) == 0.0
assert float(higher.loc['seed_ew']) > 0.5

       book window_end  factor_share_diagonal  factor_share_realized  total_variance  factor_variance_diagonal  idio_variance_diagonal  idio_variance_realized  n_names_covariance
    seed_ew 2015-01-30               0.994776               0.997030        0.000029                  0.000029                0.000000                0.000000                 444
    seed_ew 2015-02-27               0.993985               0.996761        0.000027                  0.000027                0.000000                0.000000                 445
    seed_ew 2015-03-31               0.994637               0.996801        0.000027                  0.000027                0.000000                0.000000                 445
    seed_ew 2015-04-30               0.993821               0.996616        0.000026                  0.000025                0.000000                0.000000                 445
    seed_ew 2015-05-29               0.993346               0.995867        0.000022                  0.0

/var/folders/ts/d09tttfn5lv78xjnbv8vqn_r0000gn/T/ipykernel_87651/2582906894.py:1: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(residual.round(6).to_string(index=False))


## 8. The bias statistic

F3.6 asks for the monthly bias statistic over 2015 to 2026 for both seed
books. The reading is the average of the monthly statistics, the reading E2
used for F2.4, and the monthly distribution is stored beside it. The criterion
was pre-registered as expected to fail for the momentum book.

In [22]:
# the predicted volatility is point in time: the covariance at each month end
# uses only factor returns up to that date
sample = bias.loc[bias['book'] == 'seed_ew'].iloc[-6:]
print(sample[['date', 'predicted_vol_ann', 'realized_vol_ann', 'bias_ratio']].to_string(index=False))
print()
print("predicted volatility column is the model's annualized sigma_p at that date;")
print("realized is the book's own next-21-session standard deviation, annualized.")
assert (bias['predicted_vol_ann'] > 0).all()

      date  predicted_vol_ann  realized_vol_ann  bias_ratio
2026-04-30           0.138080          0.171269    1.240357
2026-05-29           0.122413          0.101073    0.825670
2026-06-30           0.117273          0.100427    0.856351
2026-07-31           0.112124          0.122956    1.096608
2026-08-31           0.110400          0.090168    0.816734
2026-09-03           0.110939          0.085313    0.769011

predicted volatility column is the model's annualized sigma_p at that date;
realized is the book's own next-21-session standard deviation, annualized.


## 8a. The F3.6 failure on the momentum book, and which part of the model causes it

INPUT: `eval/xs_bias.parquet` (book, date, predicted_vol_ann, realized_vol_ann,
bias_ratio), `eval/xs_bias_by_exposure.parquet` (book, bucket, exposure_mean,
predicted_vol_ann_mean, realized_vol_ann_mean, bias_mean, share_in_band) and the
stored F3.6 and F3.8 numbers in `sprints/E3/RESULTS.json`.

E2's 1.8544 and XS-v1's 0.6664 are not two estimates of one quantity, so the
disagreement is not evidence that either is wrong. E2's statistic asks whether
the book's returns are explained by the factor exposures E2 measured at
full-sample name-level betas: a book re-sorted every month has a momentum
exposure that varies, E2's static betas average that variation to roughly zero,
so E2's model looks too small and the realized volatility comes out above it.
XS-v1's statistic asks whether a point-in-time covariance, built from factor
returns up to the date and the specific variance of the date, reproduces the
book's next 21 sessions. Different model, different regressors, different
window, different answer, and the second one is the model being registered.

The diagnostic that separates the two candidate causes is to split the book's
months by the book's own XS-v1 momentum exposure. If the misestimate is
concentrated in the high-exposure bucket, the factor covariance is the suspect,
because only the factor term responds to the book's own exposure. If it is flat
across the buckets, the specific variance is the suspect, because that is the
term which does not. The table stored by this sprint answers it.


In [23]:
# INPUT: data/eval/xs_bias_by_exposure.parquet (book, factor, bucket, n_months,
#        exposure_min, exposure_mean, exposure_max, predicted_vol_ann_mean,
#        realized_vol_ann_mean, bias_mean, bias_median, share_in_band) and
#        data/eval/xs_bias.parquet (book, date, bias_ratio)
# OUTPUT: none, printed and asserted only
bucket_table = pd.read_parquet(DATA / 'eval' / 'xs_bias_by_exposure.parquet')
print(f"terciles of each book's own momentum exposure, {bucket_table['n_months'].sum()} months")
print(bucket_table.round(6).to_string(index=False))
print()
pivot = bucket_table.pivot(index='bucket', columns='book',
                           values=['bias_mean', 'predicted_vol_ann_mean',
                                   'realized_vol_ann_mean'])
print(pivot.loc[['low', 'mid', 'high']].round(6).to_string())

mom = bucket_table.loc[bucket_table['book'] == 'seed_mom_ls'].set_index('bucket')
ew = bucket_table.loc[bucket_table['book'] == 'seed_ew'].set_index('bucket')
print()
print("momentum book, bucket by bucket")
for bucket in ('low', 'mid', 'high'):
    row = mom.loc[bucket]
    print(f"  {bucket:4s} exposure {row['exposure_mean']:+.4f} "
          f"predicted {row['predicted_vol_ann_mean']:.4f} "
          f"realized {row['realized_vol_ann_mean']:.4f} bias {row['bias_mean']:.4f}")
print("equal-weight book, bucket by bucket")
for bucket in ('low', 'mid', 'high'):
    row = ew.loc[bucket]
    print(f"  {bucket:4s} exposure {row['exposure_mean']:+.4f} "
          f"predicted {row['predicted_vol_ann_mean']:.4f} "
          f"realized {row['realized_vol_ann_mean']:.4f} bias {row['bias_mean']:.4f}")

# the momentum book's bias falls monotonically with its own exposure while the
# model's prediction does not move: the factor term is the one that fails to
# respond to the book, which is the diagnosis the criterion needs
assert list(mom.index) == ['low', 'mid', 'high']
assert mom['exposure_mean'].is_monotonic_increasing
assert mom.loc['high', 'bias_mean'] < mom.loc['mid', 'bias_mean'] < mom.loc['low', 'bias_mean']
assert mom.loc['high', 'realized_vol_ann_mean'] < mom.loc['low', 'realized_vol_ann_mean']
predicted_swing = float(mom['predicted_vol_ann_mean'].max() - mom['predicted_vol_ann_mean'].min())
realized_swing = float(mom['realized_vol_ann_mean'].max() - mom['realized_vol_ann_mean'].min())
print()
print(f"predicted swing across buckets {predicted_swing:.6f}, "
      f"realized swing {realized_swing:.6f}")
assert predicted_swing < realized_swing, (
    "the model's prediction does not respond to the book's own exposure"
)
# the equal-weight book has no exposure to speak of, and its bias is flat
assert abs(float(ew['bias_mean'].max() - ew['bias_mean'].min())) < 0.1

stored_bias = results['criteria']['F3.6']['stored_numbers']
stored_resid = results['criteria']['F3.8']['stored_numbers']
print()
print("stored F3.6 means:", {book: round(values['mean'], 6)
                             for book, values in stored_bias.items()})
print("stored F3.8 factor shares, momentum book:",
      stored_resid['seed_mom_ls']['factor_share_diagonal_mean'],
      stored_resid['seed_mom_ls']['factor_share_realized_mean'])
assert stored_bias['seed_mom_ls']['mean'] == approx(
    float(bias.loc[bias['book'] == 'seed_mom_ls', 'bias_ratio'].mean()), rel=1e-12
)
assert abs(float(bias.loc[bias['book'] == 'seed_mom_ls', 'bias_ratio'].mean())
           - float(mom['bias_mean'].mean())) < 1e-12
assert stored_resid['seed_mom_ls']['factor_share_diagonal_mean'] > 0
print()
print("Reading. The bias falls from the low bucket to the high bucket while the")
print("model's predicted volatility is flat across them, so the misestimate lives")
print("in the factor term: an unconditional factor covariance prices a book with a")
print("large momentum tilt the same as a book with a small one, and the realized")
print("risk of the two is not the same. The specific variance is separately")
print("mis-specified in the other direction, which F3.8 already shows: the diagonal")
print("D leaves the momentum book a factor share of "
      f"{stored_resid['seed_mom_ls']['factor_share_diagonal_mean']:.6f} against")
print(f"{stored_resid['seed_mom_ls']['factor_share_realized_mean']:.6f} with the realized residual covariance,")
print("so D understates that book's idiosyncratic risk. Both findings are about the")
print("same assumption: a covariance estimated unconditionally cannot price a book")
print("whose exposure moves. The first thing E4 should test is a covariance")
print("conditioned on the book's own exposure at the rebalance, with the third")
print("bucket above as the acceptance test: the prediction has to move with the")
print("exposure, and the bias has to flatten across the three buckets.")


terciles of each book's own momentum exposure, 282 months
       book   factor bucket  n_months  exposure_mean  exposure_min  exposure_max  predicted_vol_ann_mean  realized_vol_ann_mean  bias_mean  bias_median  share_in_band
    seed_ew momentum    low        47      -0.069807     -0.101861     -0.053171                0.145896               0.128612   0.941298     0.694020       0.255319
    seed_ew momentum    mid        47      -0.040590     -0.052729     -0.030573                0.115005               0.105872   0.950207     0.987708       0.425532
    seed_ew momentum   high        47      -0.009861     -0.030097      0.026844                0.132865               0.128373   0.979524     0.856351       0.425532
seed_mom_ls momentum    low        47       0.561474      0.231959      0.664148                0.054741               0.047864   0.895986     0.691513       0.255319
seed_mom_ls momentum    mid        47       0.731709      0.670813      0.796305                0.058397   

## 9. What E3 inherits and what it hands on

| Item | Value | Where |
| --- | --- | --- |
| MODEL_START, time-series | 2010 | E2, ledger 2026-09-10 |
| XS-v1 start | see F3.1 stored n_days and the registry model_start | registry |
| Exclusion list | 33 tickers plus 3 truncated, F2.6b and F2.6c | TS-v1 registry |
| Survivor-only cross-section | 41.9 percent of the 2010 index by name, 8.85 by cap | PROBES, E4 open item |
| Share counts | as filed from 2013-04, dense from 2015-10, 1,245,448 look-ahead name-days | ledger 2026-09-17 |
| Production volatility | EWMA(0.97) | E2 |
| Realized residual covariance | F3.8, both numbers stored | this sprint |
| Conditional exposure | F3.9, exposure series stored | this sprint |
| Champion rule | unchanged, XS-v1 eligible but not champion | registry |

Open items handed to E4: the survivor-only cross-section needs a point-in-time
sector and constituent source, and the momentum book's risk needs a conditional
covariance before it can be sized. Both are recorded in docs/open_items.md.

In [24]:
print("credit port note")
print("In credit the cross-sectional regressors become spread duration, credit")
print("quality buckets instead of GICS sectors, and issuer-level liquidity; the")
print("estimator is unchanged. Two findings should be larger there:")
print("  1. Identity: bonds carry CUSIPs but issuers merge, are re-tranched and")
print("     are called, so the ticker identity problem is worse, not better.")
print("  2. Staleness: most bonds do not trade daily, so the return panel is")
print("     sparse and the descriptor windows have to be defined on non-traded")
print("     days, which is the stale-flag problem at a much larger scale.")
print("  3. Exposure timing: a book sorted on spread has the same dynamic")
print("     exposure problem C8 found for momentum, and the same fix is needed.")
print()
print("Credit port, the E3 close-out addition:")
print("In credit the descriptor set becomes spread duration or DTS, spread level,")
print("rating and sector dummies, issue size, age, liquidity and spread momentum,")
print("and the identification constraint carries over unchanged: cap or notional")
print("weighted sector or rating factor returns still sum to zero, so the")
print("reference bucket is still dropped and its return still derived.")
print("Of E3's problems, two change shape and one gets worse. The sector or rating")
print("file is more point-in-time in credit, because index files exist, so the")
print("survivor problem of section 9a should shrink rather than grow. The per-day")
print("cross-section is far thinner, which makes the fifteen to twenty factor")
print("design harder to estimate on a given day and makes the rank-deficiency case")
print("routine rather than exceptional. And the exposure timing problem of section")
print("8a is exactly the problem a spread-sorted book has: a book re-sorted on")
print("spread carries a dynamic exposure to the spread factor, so an unconditional")
print("covariance will not price it, and the fix is the same conditional covariance")
print("E4 owes the momentum book here.")


credit port note
In credit the cross-sectional regressors become spread duration, credit
quality buckets instead of GICS sectors, and issuer-level liquidity; the
estimator is unchanged. Two findings should be larger there:
  1. Identity: bonds carry CUSIPs but issuers merge, are re-tranched and
     are called, so the ticker identity problem is worse, not better.
  2. Staleness: most bonds do not trade daily, so the return panel is
     sparse and the descriptor windows have to be defined on non-traded
     days, which is the stale-flag problem at a much larger scale.
  3. Exposure timing: a book sorted on spread has the same dynamic
     exposure problem C8 found for momentum, and the same fix is needed.

Credit port, the E3 close-out addition:
In credit the descriptor set becomes spread duration or DTS, spread level,
rating and sector dummies, issue size, age, liquidity and spread momentum,
and the identification constraint carries over unchanged: cap or notional
weighted sector or r

## 8b. Coverage qualifies the equal-weight book's pass

INPUT: `eval/xs_coverage_by_year.parquet` (book, year, n_months, coverage_mean,
coverage_min, coverage_max, n_names_mean, bias_mean, share_in_band) and
`eval/xs_risk_decomposition.parquet` (book, date, residual_weight, n_names).

F3.6 passes on the equal-weight book at 0.9570, but the model describes between
58 and 98 percent of that book's weight depending on the year. Coverage is one
minus the residual weight, the weight of the book's names with no descriptor
row, and it is the reason the number has to be quoted with its period attached:
an in-band mean over years that are two thirds covered is the bias of the part
of the book the model can see, not the bias of the book.


In [25]:
# INPUT: data/eval/xs_coverage_by_year.parquet (book, year, coverage_mean,
#        coverage_min, coverage_max, n_names_mean, bias_mean, share_in_band) and
#        data/eval/xs_risk_decomposition.parquet (book, date, residual_weight)
# OUTPUT: none, printed and asserted only
coverage = pd.read_parquet(DATA / 'eval' / 'xs_coverage_by_year.parquet')
print("bias beside coverage, by year")
print(coverage.round(6).to_string(index=False))

by_date = (decomposition.groupby(['book', 'date'], as_index=False)
           .agg(residual_weight=('residual_weight', 'first')))
by_date['year'] = pd.to_datetime(by_date['date']).dt.year
full_span = (by_date.assign(coverage=1.0 - by_date['residual_weight'])
             .groupby(['book', 'year'])['coverage'].mean().round(4).unstack(0))
print()
print("coverage by year over the whole model span, from the decomposition artifact")
print(full_span.to_string())

ew_cov = coverage.loc[coverage['book'] == 'seed_ew'].set_index('year')['coverage_mean']
mom_cov = coverage.loc[coverage['book'] == 'seed_mom_ls'].set_index('year')['coverage_mean']
print()
print(f"seed_ew coverage {ew_cov.iloc[0]:.6f} in {ew_cov.index[0]} to "
      f"{ew_cov.iloc[-1]:.6f} in {ew_cov.index[-1]}")
print(f"seed_mom_ls coverage {mom_cov.iloc[0]:.6f} to {mom_cov.iloc[-1]:.6f}")
assert ew_cov.iloc[0] < 0.7, "the first bias year is only partly covered"
assert ew_cov.is_monotonic_increasing, "coverage rises as the book moves inside the universe"
assert abs(float(mom_cov.mean()) - 1.0) < 0.01, "the momentum book is inside the universe"
assert ew_cov.iloc[-1] > 0.95
# the whole span starts lower still: 2011 is the worst year of the model
ew_span = full_span['seed_ew']
print()
print(f"seed_ew coverage over the model span: {ew_span.index[0]} "
      f"{ew_span.iloc[0]:.4f} to {ew_span.index[-1]} {ew_span.iloc[-1]:.4f}")
assert ew_span.iloc[0] < 0.6, "the first model year is under 60 percent covered"
print()
print("Reading. The pass on the equal-weight book is a pass for the part of the")
print("book the model describes. In the early years that is a minority of the")
print("weight, so the in-band mean and the low-coverage years are one statement:")
print("the model prices what it can see, and what it can see is a survivor-only")
print("cross-section. E4's point-in-time sector and constituent source is what")
print("turns these two series into one.")


bias beside coverage, by year
       book  year  n_months  coverage_mean  coverage_min  coverage_max  n_names_mean  predicted_vol_ann_mean  realized_vol_ann_mean  bias_mean  share_in_band
    seed_ew  2015        12       0.653972      0.646000      0.663347    327.916667                0.090269               0.072750   0.797449       0.416667
    seed_ew  2016        12       0.685242      0.665339      0.700397    345.083333                0.104862               0.102713   1.000360       0.250000
    seed_ew  2017        12       0.724372      0.702381      0.740079    365.083333                0.068536               0.086365   1.247854       0.583333
    seed_ew  2018        12       0.750827      0.742063      0.761905    378.416667                0.085859               0.052658   0.630602       0.250000
    seed_ew  2019        12       0.769772      0.761905      0.788538    389.250000                0.110075               0.110691   1.012453       0.250000
    seed_ew  2020     

## 8c. The mean passes while most months miss

INPUT: `eval/xs_bias.parquet` (book, date, bias_ratio) and the stored F3.6
numbers in `sprints/E3/RESULTS.json`.

A mean inside the band is not the same claim as a book that is priced correctly
month after month. The criterion is written on the mean, so the distribution is
printed beside it, and the distribution is what a risk limit is actually written
against.


In [26]:
# INPUT: data/eval/xs_bias.parquet (book, date, bias_ratio, predicted_vol_ann,
#        realized_vol_ann) and the stored F3.6 numbers in sprints/E3/RESULTS.json
# OUTPUT: none, printed and asserted only
rows = []
for book, group in bias.groupby('book'):
    values = group['bias_ratio'].dropna()
    rows.append({
        'book': book,
        'months': len(values),
        'mean': float(values.mean()),
        'p10': float(values.quantile(0.10)),
        'median': float(values.median()),
        'p90': float(values.quantile(0.90)),
        'share_in_band': float(values.between(0.8, 1.25).mean()),
        'share_below': float((values < 0.8).mean()),
        'share_above': float((values > 1.25).mean()),
    })
distribution = pd.DataFrame(rows).set_index('book')
print(distribution.round(6).to_string())

stored = results['criteria']['F3.6']['stored_numbers']
print()
for book, summary in sorted(stored.items()):
    print(f"  stored {book}: mean {summary['mean']:.6f}, "
          f"share in band {summary['share_in_band']:.6f}, months {summary['n_months']}")
    assert summary['mean'] == approx(float(distribution.loc[book, 'mean']), rel=1e-12)
    assert summary['share_in_band'] == approx(
        float(distribution.loc[book, 'share_in_band']), rel=1e-12
    )
# the seed_ew mean is inside the band while two thirds of its months are not
assert 0.8 <= float(distribution.loc['seed_ew', 'mean']) <= 1.25
assert float(distribution.loc['seed_ew', 'share_in_band']) < 0.5
assert float(distribution.loc['seed_mom_ls', 'mean']) < 0.8
print()
print("Reading. The equal-weight book passes on the mean and fails on the")
print("distribution: its 10th percentile is far below the band and its 90th only")
print("reaches it, so the same model that sizes the book correctly on average is")
print("wrong by a large factor in individual months, and it is wrong in both")
print("directions in different regimes. That is the difference between a model")
print("that can rank risk and a model that can size a position, and closing it is")
print("E5's bias work rather than E3's. The momentum book needs B's conditional")
print("covariance first.")


             months      mean       p10    median       p90  share_in_band  share_below  share_above
book                                                                                                
seed_ew         141  0.957010  0.457442  0.873245  1.531699       0.368794     0.439716     0.191489
seed_mom_ls     141  0.666387  0.308023  0.559259  1.096834       0.120567     0.808511     0.070922

  stored seed_ew: mean 0.957010, share in band 0.368794, months 141
  stored seed_mom_ls: mean 0.666387, share in band 0.120567, months 141

Reading. The equal-weight book passes on the mean and fails on the
distribution: its 10th percentile is far below the band and its 90th only
reaches it, so the same model that sizes the book correctly on average is
wrong by a large factor in individual months, and it is wrong in both
directions in different regimes. That is the difference between a model
that can rank risk and a model that can size a position, and closing it is
E5's bias work rather 

## 9a. The survivor-only cross-section, as a number

INPUT: `eval/xs_survivor_restriction.parquet` (year, n_days, n_members_mean,
n_mapped_mean, n_outside_mean, n_outside_distinct, share_outside,
share_outside_mcap), the stored probe values in `sprints/E3/PROBES.md` and
`docs/open_items.md`.

The sector dummies restrict the cross-section to the 502 names the sector file
holds, and that file is a current-member snapshot, so the historical
cross-section contains only companies in the index today. The restriction is
measured two ways because they answer different questions: the share of index
member names it removes, and the share of index market capitalisation.


In [27]:
# INPUT: data/eval/xs_survivor_restriction.parquet (year, n_members_mean,
#        n_mapped_mean, n_outside_mean, share_outside, share_outside_mcap)
# OUTPUT: none, printed and asserted only
survivors = pd.read_parquet(DATA / 'eval' / 'xs_survivor_restriction.parquet')
print(survivors[['year', 'n_members_mean', 'n_mapped_mean', 'n_outside_mean',
                 'n_outside_distinct', 'share_outside', 'share_outside_mcap']]
      .round(6).to_string(index=False))

first, last = survivors.iloc[0], survivors.iloc[-1]
print()
print(f"{int(first['year'])}: {int(first['n_outside_mean'])} of "
      f"{int(round(first['n_members_mean']))} members outside the sector file, "
      f"share by name {first['share_outside']:.6f}, by market cap {first['share_outside_mcap']:.6f}")
print(f"{int(last['year'])}: {int(last['n_outside_mean'])} of "
      f"{int(round(last['n_members_mean']))} members outside, "
      f"share by name {last['share_outside']:.6f}, by market cap {last['share_outside_mcap']:.6f}")
print(f"mapped names per day {survivors['n_mapped_mean'].iloc[0]:.1f} at the start and "
      f"{survivors['n_mapped_mean'].iloc[-1]:.1f} at the end, out of "
      f"{survivors['n_members_mean'].iloc[-1]:.1f} members")

# the arithmetic identity of the table: mapped plus outside is the member count
assert np.allclose(
    survivors['n_mapped_mean'] + survivors['n_outside_mean'], survivors['n_members_mean']
)
assert survivors['share_outside'].is_monotonic_decreasing
assert first['share_outside'] > 0.4
assert last['share_outside'] < 0.05
# the market-cap share is the smaller number in every year, because the names
# that leave are small ones
assert (survivors['share_outside_mcap'] < survivors['share_outside']).all()
assert first['share_outside_mcap'] < 0.10
assert last['share_outside_mcap'] < 0.01
print()
print("Reading. The restriction removes 41.9 percent of the 2010 index by name and")
print("8.9 percent by market capitalisation, and both decay as the missing names")
print("leave. This is a stronger restriction than the 366 basis points a year")
print("F1.5 measures on the full panel, because F1.5 is estimated from the members")
print("the panel does hold while the sector dummies remove the members it does not")
print("hold at all. The direction of the bias is knowable and worth stating: a")
print("survivor-only cross-section flatters the premia, flatters the R squared and")
print("flatters the specific risk, because the failures are missing rather than")
print("badly estimated. It is E4's to fix and not E3's, and no criterion in this")
print("sprint is written on a number it could move.")


 year  n_members_mean  n_mapped_mean  n_outside_mean  n_outside_distinct  share_outside  share_outside_mcap
 2010      503.876984     292.555556      211.321429                 219       0.419391            0.088511
 2011      502.202381     298.146825      204.055556                 215       0.406321            0.089034
 2012      502.976000     306.848000      196.128000                 206       0.389935            0.072907
 2013      501.845238     316.773810      185.071429                 196       0.368782            0.069569
 2014      500.484127     323.083333      177.400794                 188       0.354458            0.075068
 2015      502.317460     332.706349      169.611111                 188       0.337657            0.054653
 2016      503.738095     350.321429      153.416667                 175       0.304556            0.049406
 2017      504.000000     367.832669      136.167331                 155       0.270173            0.042654
 2018      504.000000     38

## 9b. The look-ahead flag is in the weights, not only in the Size descriptor

INPUT: `processed/market_cap.parquet` (date, ticker, close, shares,
market_cap, look_ahead), `models/XS-v1/factor_returns.parquet` (date, factor, f)
and the share-count row in the inheritance table above.

The pre-2015 flag is usually described as a Size descriptor problem, and that
description is too narrow. The estimator weights every observation by the square
root of market capitalisation, and market capitalisation is close times shares.
A backfilled share count therefore enters the weights as well as the
descriptor, so a sensitivity re-estimated from 2015-10-23 prices both: it
replaces the projection in the Size column and it drops the stale count from the
weights. The two factor return windows below show that the effect is not
confined to Size.


In [28]:
# INPUT: data/processed/market_cap.parquet (date, ticker, market_cap, look_ahead)
#        and data/models/XS-v1/factor_returns.parquet (date, factor, f)
# OUTPUT: none, printed and asserted only
flag = market_cap.loc[:, ['date', 'ticker', 'market_cap', 'look_ahead']].copy()
flag['year'] = pd.to_datetime(flag['date']).dt.year
flagged = (flag.assign(weight=flag['market_cap'].where(flag['market_cap'] > 0))
           .groupby('year')
           .apply(lambda block: pd.Series({
               'names': len(block),
               'share_names_flagged': float(block['look_ahead'].mean()),
               'share_weight_flagged': float(
                   block.loc[block['look_ahead'], 'weight'].sum()
                   / block['weight'].sum()
               ),
           }), include_groups=False))
print("the flag in names and in weight, by year")
print(flagged.round(6).to_string())

pre = flag.loc[flag['year'] < 2015, 'look_ahead'].mean()
post = flag.loc[flag['year'] >= 2016, 'look_ahead'].mean()
print()
print(f"share of name-days flagged before 2015 {pre:.6f}, from 2016 {post:.6f}")
assert pre > 0.99, "almost every pre-2015 name-day is a projection"
assert post < 0.10, "the flag is mostly gone once the vendor history is dense"
assert post < pre / 10.0, "the drop is an order of magnitude"
# once the history is dense the flag that remains is the names the vendor
# never answered for, and those names are smaller than average
assert (
    flagged.loc[2016, 'share_weight_flagged']
    < flagged.loc[2016, 'share_names_flagged']
)
# the weight share is what the sqrt(mcap) estimator sees, and it is not zero
assert float(flagged.loc[2011, 'share_weight_flagged']) > 0.9
assert float(flagged.loc[2025, 'share_weight_flagged']) < 0.05

wide = (factor_returns.pivot(index='date', columns='factor', values='f'))
flagged_window = wide.loc[:'2015-10-22']
clean_window = wide.loc['2015-10-23':]
print()
print(f"factor return windows: flagged {flagged_window.index.min().date()} to "
      f"{flagged_window.index.max().date()} ({len(flagged_window)} days)")
print(f"                       clean   {clean_window.index.min().date()} to "
      f"{clean_window.index.max().date()} ({len(clean_window)} days)")
rows = []
for factor in ('size', 'beta', 'momentum', 'market'):
    rows.append({
        'factor': factor,
        'flagged mean (annualized)': float(flagged_window[factor].mean() * 252),
        'clean mean (annualized)': float(clean_window[factor].mean() * 252),
        'flagged vol (annualized)': float(flagged_window[factor].std(ddof=1) * np.sqrt(252)),
        'clean vol (annualized)': float(clean_window[factor].std(ddof=1) * np.sqrt(252)),
    })
comparison = pd.DataFrame(rows).set_index('factor')
print()
print(comparison.round(6).to_string())
size_row = comparison.loc['size']
assert not np.isclose(
    float(size_row['flagged mean (annualized)']), float(size_row['clean mean (annualized)'])
), "Size is not the only thing the flag moves"
moved = int(
    ((comparison['flagged vol (annualized)']
      - comparison['clean vol (annualized)']).abs() > 0.01).sum()
)
print()
print(f"{moved} of {len(comparison)} factors printed have a volatility that moves by")
print("more than one annual percentage point between the two windows, which is why")
print("the sensitivity is priced from 2015-10-23 as a re-estimation rather than as")
print("a Size-only switch. Before that date the weights are built on share counts")
print("whose filing date is later than the date they are used on, which is the")
print("same look-ahead the descriptor carries and the reason the whole sensitivity")
print("starts at the date the vendor history becomes dense.")


the flag in names and in weight, by year
         names  share_names_flagged  share_weight_flagged
year                                                     
2010  126504.0             1.000000              1.000000
2011  126504.0             1.000000              1.000000
2012  125500.0             1.000000              1.000000
2013  126504.0             0.997146              0.996471
2014  126504.0             0.992095              0.994154
2015  126504.0             0.853017              0.842317
2016  126504.0             0.129182              0.069707
2017  126002.0             0.113824              0.069325
2018  126002.0             0.100998              0.063417
2019  126504.0             0.086227              0.052405
2020  127006.0             0.063438              0.043609
2021  126504.0             0.048441              0.041535
2022  126002.0             0.039166              0.017989
2023  125500.0             0.029339              0.006223
2024  126504.0             0.01

## 10. Every criterion

Loop over the stored file rather than a hardcoded list, and assert the loop
covers every criterion in it.

In [29]:
rows = []
for name, block in sorted(results['criteria'].items()):
    stored = block['stored_numbers']
    key = next((k for k, v in stored.items() if isinstance(v, (int, float))), None)
    rows.append({
        'criterion': name,
        'verdict': block['verdict'],
        'threshold': block['threshold'],
        'stored number': (f"{key} = {stored[key]}" if key else 'composite'),
    })
table = pd.DataFrame(rows)
print(table.to_string(index=False))
print()
assert set(table['criterion']) == {f"F3.{i}" for i in range(1, 10)}
assert len(table) == 9
print("failures:", list(table.loc[table['verdict'] == 'fail', 'criterion']))
print("every criterion in the file is covered by this notebook")

criterion verdict                                                   threshold                                             stored number
     F3.1    pass mean cross-sectional R squared > 0.20, fail band below 0.15       mean_cross_sectional_r_squared = 0.3294501878861149
     F3.2    pass                           max abs deviation from e_k < 1e-8 max_abs_identity_error_estimated = 1.2426149519073615e-13
     F3.3    pass            abs cap-weighted sector sum < 1e-10 on every day   max_abs_cap_weighted_sector_sum = 7.047314121155779e-18
     F3.4    pass     momentum correlation > 0.6 and market correlation > 0.9       momentum_vs_ff_mom_correlation = 0.7555482343048963
     F3.5    pass                         identity error at machine precision                max_abs_factor_plus_idio_minus_total = 0.0
     F3.6    fail             mean monthly bias in [0.8, 1.25] for both books                                                 composite
     F3.7    pass               table complete f

In [30]:
for name, block in sorted(results['criteria'].items()):
    print(f"{name} ({block['verdict']}): {block['note']}")

F3.1 (pass): The three diagnostics standing instruction B names are stored here whether or not the average clears 20 percent, so a low number can be attributed without a rebuild. The shift test pair is stored beside them: the design dated t-1 explains shift_test_lagged_mean_r_squared and the design dated t explains shift_test_dated_t_mean_r_squared, and the difference is the same-day information the t-1 rule removes.
F3.2 (pass): The criterion tests the factor-mimicking portfolios of the estimated design, the rows of (X'WX)^-1 X'W, where the reference sector is dropped so the market column and the sector block are not collinear. The identified weight set reproduces all eleven reported sector returns and is stored beside it.
F3.3 (pass): Measured on the identified factor returns, every day.
F3.4 (pass): The market comparison is against Mkt-RF plus RF, the FF total return, on the overlap through 2026-07-31. XS-v1 regresses the total return r because the risk-free series ends there.
F3.5 

## 11. Dashboard D2 map

Every panel and the parquet column it reads.

| Panel | Artifact | Columns |
| --- | --- | --- |
| factor returns, daily and cumulative | models/XS-v1/factor_returns.parquet | date, factor, f |
| premia and t statistics | eval/xs_fm_premia.parquet | factor, period, premium_annualized, nw_se, t_stat, priced |
| R squared series | models/XS-v1/xs_r2.parquet | date, r_squared, fmp_identity_max_abs_error, n_names |
| descriptor coverage and distributions | models/XS-v1/descriptors.parquet | date, descriptor, value_z |
| factor covariance and correlation | models/XS-v1/factor_cov.parquet | factor names as index and columns |
| FMP explorer | models/XS-v1/fmp_weights.parquet | date, factor, ticker, weight, kind |
| risk decomposition | eval/xs_risk_decomposition.parquet | book, date, level, name, exposure, mcr, contribution |
| exposure timing | eval/xs_exposure_timeseries.parquet | date, factor, exposure |
| idio share both ways | eval/xs_residual_covariance.parquet | book, factor_share_diagonal, factor_share_realized |

In [31]:
from dashboard.tabs import d02_factor_risk as d2

checks = {
    'factor returns': d2.factor_return_chart(factor_returns, 'momentum'),
    'premia': d2.premia_table(premia, 'full_sample'),
    'r squared': d2.r_squared_series(xs_r2),
    'coverage': d2.descriptor_coverage(descriptors),
    'correlation': d2.factor_correlation(factor_cov),
    'fmp': d2.fmp_explorer(fmp, 'momentum', 'estimated'),
    'risk table': d2.risk_table(decomposition, 'seed_mom_ls'),
    'mcr': d2.mcr_chart(decomposition, 'seed_mom_ls'),
    'exposure': d2.exposure_timing(exposure, 'seed_mom_ls', 'momentum'),
    'residual': d2.residual_covariance_panel(residual),
}
for name, frame in checks.items():
    print(f"{name}: {frame.shape[0]} rows x {frame.shape[1]} columns")
    assert not frame.empty, name
print("every D2 panel builder returns rows from the current artifacts")

2026-09-17 22:24:21.055 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


factor returns: 3941 rows x 2 columns
premia: 18 rows x 6 columns
r squared: 3941 rows x 4 columns
coverage: 189 rows x 7 columns
correlation: 18 rows x 18 columns
fmp: 20 rows x 1 columns
risk table: 18 rows x 2 columns
mcr: 20 rows x 15 columns
exposure: 141 rows x 1 columns
residual: 2 rows x 2 columns
every D2 panel builder returns rows from the current artifacts


## 12. Closing checklist

In [32]:
stored_values = []


def collect(node):
    if isinstance(node, dict):
        for value in node.values():
            collect(value)
    elif isinstance(node, list):
        for value in node:
            collect(value)
    elif isinstance(node, float):
        stored_values.append(node)


collect(results['criteria'])
collect(entry['parameters'])
# the E3 close-out stores three more tables, whose values are stored
# values too, so they join the scan
for frame in (bias_by_exposure_table, coverage_table, survivor_table):
    for column in frame.select_dtypes('number').columns:
        collect([float(value) for value in frame[column]])

print(f"stored values collected from RESULTS.json and the registry: {len(stored_values)}")

import json as _json

notebook = _json.loads((ROOT / 'notebooks' / 'E3_walkthrough.ipynb').read_text())
code_cells = [c for c in notebook['cells'] if c['cell_type'] == 'code']
literal_hits = []
for cell in code_cells:
    source = ''.join(cell['source'])
    for value in stored_values:
        for text in {f"{value:.6f}", f"{value:.4f}", repr(value)}:
            if len(text) > 6 and text in source:
                literal_hits.append(text)
print("stored values typed into a code cell:", sorted(set(literal_hits))[:5] or "none")
assert not literal_hits, "a stored value was typed into the notebook instead of read"

# the assert count in this notebook, against the count before the six E3
# close-out sections were added
previous_asserts = 40
assert_count = sum(''.join(c['source']).count('assert ') for c in code_cells)
print()
print("closing checklist")
print(f"  criteria covered by section 10: {len(results['criteria'])} of 9")
assert len(results['criteria']) == 9
print(f"  asserts in this notebook: {assert_count}, every one of them passed")
print(f"  asserts before the E3 close-out: {previous_asserts}, "
      f"added {assert_count - previous_asserts}")
print(f"  code cells: {len(code_cells)}")
assert assert_count > previous_asserts
print(f"  data hash the notebook ran against: {version['data_hash']}")
print(f"  XS-v1 artifacts hash: {entry['parameters']['artifacts_hash']}")
print(f"  no stored number hardcoded: {not literal_hits}")
print(f"  verdicts: " + ", ".join(f"{k} {v['verdict']}" for k, v in sorted(results['criteria'].items())))


stored values collected from RESULTS.json and the registry: 571
stored values typed into a code cell: none

closing checklist
  criteria covered by section 10: 9 of 9
  asserts in this notebook: 86, every one of them passed
  asserts before the E3 close-out: 40, added 46
  code cells: 32
  data hash the notebook ran against: 237c1183b898a054118c1dd0e7fbfa0556b9162841d9eb7fe242da7fa14d2b36
  XS-v1 artifacts hash: 87d4d6af68bc4ce60c791405a954b876b66c41ee3e34349d619a93329ec58e83
  no stored number hardcoded: True
  verdicts: F3.1 pass, F3.2 pass, F3.3 pass, F3.4 pass, F3.5 pass, F3.6 fail, F3.7 pass, F3.8 pass, F3.9 pass
